# data

> Data loading, tokenization, and dataset assembly for fastpragma

All internal processing uses polars.

In [ ]:
#| default_exp data

In [ ]:
#| export
import polars as pl, numpy as np, pandas as pd, os, json, tempfile
from datetime import datetime, timezone
from dataclasses import dataclass, field
from collections import defaultdict
from fastcore.all import *
from fastai.data.external import untar_data, URLs
from tokenizers import Tokenizer as HFTokenizer, models, pre_tokenizers, trainers, decoders
from fastprogress.fastprogress import progress_bar

In [ ]:
path = untar_data(URLs.ML_100k)

## SourceSchema

Declare each data source with its column types. Profile sources omit `time_col` — all fields share the eval-point timestamp. Event sources provide a `time_col` for ordering.

In [ ]:
#| export
#| export
class DataSource:
    def __init__(self, 
                df: pl.LazyFrame, 
                cats: list = None, 
                conts: list = None,
                signed_conts:list=None,
                texts: list = None,
                time_col: str = None,
                lifelong: list = None,
                num_buckets: int = 100,
                cardinality_threshold: int = 100,
                entity_col: str = 'entity_id',
                is_profile: bool = False,
                name: str=None):
        schema = df.collect_schema()
        cats,conts,signed_conts,texts,lifelong = [ifnone(o,[]) for o in (cats,conts,signed_conts,texts,lifelong)]
        all_cols = {*cats, *conts, *texts, *lifelong, entity_col, *([time_col] if time_col else [])}
        self.col_names = schema.names()
        missing = all_cols - set(self.col_names)
        if missing: raise ValueError(f"Missing columns {missing}. Found columns: {self.col_names}")
        assert set(signed_conts).issubset(conts)
        store_attr()
        if is_profile:
            has_dups = df.select(pl.col(entity_col).is_duplicated().any()).collect().item()
            assert not has_dups, f"profile source must be unique by {entity_col}"


    def __repr__(self):
        return (
            f"DataSource(columns={self.col_names}, name={self.name} "
            f"cats={self.cats or '[]'}, conts={self.conts or '[]'}, "
            f"texts={self.texts or '[]'}, time_col={self.time_col!r})\n"
            f"{self.df.head().collect()}"
        )
    
    @classmethod
    def from_df(cls, df, **kwargs):
        kwargs.setdefault('name', df.attrs.get('name', 'df'))
        return cls(pl.from_pandas(df).lazy(), **kwargs)

    @classmethod
    def from_file(cls, path, **kwargs):
        path = Path(path)
        scans = dict(csv=pl.scan_csv, tsv=pl.scan_csv, parquet=pl.scan_parquet, ipc=pl.scan_ipc, feather=pl.scan_ipc)
        scan_kw_names = {'separator','has_header','new_columns','schema','schema_overrides','null_values','try_parse_dates'}
        scan_kw = {k:v for k,v in kwargs.items() if k in scan_kw_names}
        src_kw = {k:v for k,v in kwargs.items() if k not in scan_kw_names}
        src_kw.setdefault('name', path.stem)
        suffix = path.suffix[1:].lower()
        scan = scans.get(suffix, pl.scan_csv)
        if suffix == 'tsv': scan_kw = dict(separator='\t', **scan_kw)
        return cls(scan(path, **scan_kw), **src_kw)

In [ ]:
# Test 1: Explicit column types
df = pl.LazyFrame({'cat_col': ['a', 'b', 'a'], 'num_col': [1.0, 2.0, 3.0], 'id_col': [10, 20, 30]})
s = DataSource(df, cats=['cat_col'], conts=['num_col'], entity_col='id_col')
test_eq(sorted(s.cats), ['cat_col'])
test_eq(s.conts, ['num_col'])

# Test 2: Explicit overrides take priority
s2 = DataSource(df, cats=['cat_col'], conts=['num_col', 'id_col'], entity_col='id_col')
test_eq(sorted(s2.cats), ['cat_col'])
test_eq(sorted(s2.conts), ['id_col', 'num_col'])

# Test 3: time_col is excluded from cats/conts
df3 = pl.LazyFrame({'id_col': [1,2], 'evt': ['x', 'y'], 'val': [1, 2], 'ts': [100, 200]})
s3 = DataSource(df3, cats=['evt'], conts=['val'], time_col='ts', entity_col='id_col')
test_eq('ts' not in s3.cats + s3.conts, True)

# Test 4: Float32
df4 = pl.LazyFrame({'f32': [1.0, 2.0]}, schema={'f32': pl.Float32})
s4 = DataSource(df4, conts=['f32'], entity_col='f32')
test_eq(s4.conts, ['f32'])

In [ ]:
profile = DataSource.from_file(path / "u.user", separator="|", has_header=False, new_columns=["user_id","age","gender","occupation","zip_code"], 
                            cats=["gender","occupation"], conts=["age"], entity_col="user_id")
profile

DataSource(columns=['user_id', 'age', 'gender', 'occupation', 'zip_code'], name=u cats=['gender', 'occupation'], conts=['age'], texts=[], time_col=None)
shape: (5, 5)
┌─────────┬─────┬────────┬────────────┬──────────┐
│ user_id ┆ age ┆ gender ┆ occupation ┆ zip_code │
│ ---     ┆ --- ┆ ---    ┆ ---        ┆ ---      │
│ i64     ┆ i64 ┆ str    ┆ str        ┆ str      │
╞═════════╪═════╪════════╪════════════╪══════════╡
│ 1       ┆ 24  ┆ M      ┆ technician ┆ 85711    │
│ 2       ┆ 53  ┆ F      ┆ other      ┆ 94043    │
│ 3       ┆ 23  ┆ M      ┆ writer     ┆ 32067    │
│ 4       ┆ 24  ┆ M      ┆ technician ┆ 43537    │
│ 5       ┆ 33  ┆ F      ┆ other      ┆ 15213    │
└─────────┴─────┴────────┴────────────┴──────────┘

In [ ]:
pdf = pd.DataFrame({'cat': ['x','y','x'], 'val': [1,2,3], 'id': [10,20,30]})
s5 = DataSource.from_df(pdf, cats=['cat'], conts=['val'], entity_col='id')
test_eq(sorted(s5.cats), ['cat'])
test_eq(s5.conts, ['val'])
test_eq(s5.entity_col, 'id')
test_eq(s5.time_col, None)

## Key-Value-Time Conversion

Convert wide DataFrames to the uniform `(key, value, value_type, time)` format used internally. Profile rows get `time=0` (all at eval point); events use their timestamp column; lifelong columns carry their own datetime.

In [ ]:
#| export
class Tokenizer:
    def __init__(self, num_buckets=100, cardinality_threshold=100):
        store_attr()
        self.key_vocab = {'[PAD]': 0, '[MASK]': 1, '[UNK]': 2, '[USR]': 3, '[EVT]': 4}
        self.val_vocab = {'[PAD]': 0, '[MASK]': 1, '[UNK]': 2}
        self.num_bounds = {}
        self.bpe_tokenizer = None
        self.bpe_offset = 0
        self.lowcard_text_keys = set()
    def _add_key(self, k): self.key_vocab.setdefault(k, len(self.key_vocab))
    def _add_val(self, v): self.val_vocab.setdefault(v, len(self.val_vocab))

In [ ]:
#| export
@patch
def _melt_profile_fields(self:Tokenizer, src:DataSource):
    melt_cols = src.cats + src.conts + src.texts
    if not melt_cols: return pl.LazyFrame(schema=[src.entity_col, 'key', 'value', 'vtype', 'time'])
    return (
        src.df
        .select(src.entity_col, *melt_cols)
        .unpivot(index=src.entity_col, on=melt_cols, variable_name='key', value_name='value')
        .with_columns(
            pl.col('value').cast(pl.Utf8),
            pl.when(pl.col('key').is_in(src.cats)).then(pl.lit('cat'))
             .when(pl.col('key').is_in(src.conts)).then(pl.lit('num'))
             .otherwise(pl.lit('text')).alias('vtype'),
            pl.lit(0).cast(pl.Int64).alias('time')
        )
    )

In [ ]:
df = pl.LazyFrame(dict(uid=[1,2], gender=['M','F'], age=[25,30]))
src = DataSource(df, cats=['gender'], conts=['age'], entity_col='uid')
tok = Tokenizer()
res = tok._melt_profile_fields(src).collect()

test_eq(res.shape, (4,5))
test_eq(set(res.columns), {'uid','key','value','vtype','time'})
test_eq(res.filter(pl.col('key')=='gender')['vtype'].unique().to_list(), ['cat'])
test_eq(sorted(res.filter(pl.col('key')=='gender')['value'].to_list()), ['F','M'])
test_eq(res.filter(pl.col('key')=='age')['vtype'].unique().to_list(), ['num'])
test_eq(res['time'].unique().to_list(), [0])
test_eq('uid' in res['key'].to_list(), False)

src = DataSource(pl.LazyFrame(dict(uid=[1])), entity_col='uid')
test_eq(tok._melt_profile_fields(src).collect().height, 0)

In [ ]:
#| export
@patch
def _melt_events(self:Tokenizer, src:DataSource):
    if not src.time_col: raise ValueError(f"{src.name}: events require time_col")
    melt_cols = src.cats + src.conts + src.texts
    if not melt_cols: return pl.LazyFrame(schema=[src.entity_col, 'key', 'value', 'vtype', 'time', 'event_idx'])

    df = (
        src.df
        .with_row_index('_row_id')
        .sort(src.entity_col, src.time_col, '_row_id')
        .with_columns(pl.int_range(0, pl.len()).over(src.entity_col).alias('event_idx'))
    )

    time_col_type = df.collect_schema()[src.time_col]
    time_expr = (
        pl.col(src.time_col).dt.epoch(time_unit='s').cast(pl.Int64)
        if time_col_type.is_temporal()
        else pl.col(src.time_col).cast(pl.Int64)
    )

    return (
        df
        .select(src.entity_col, src.time_col, 'event_idx', *melt_cols)
        .unpivot(index=[src.entity_col, src.time_col, 'event_idx'], on=melt_cols, variable_name='key', value_name='value')
        .with_columns(
            pl.col('value').cast(pl.Utf8),
            pl.when(pl.col('key').is_in(src.cats)).then(pl.lit('cat'))
             .when(pl.col('key').is_in(src.conts)).then(pl.lit('num'))
             .otherwise(pl.lit('text')).alias('vtype'),
            time_expr.alias('time')
        )
        .drop(src.time_col)
        .sort(src.entity_col, 'event_idx')
    )

In [ ]:
df = pl.LazyFrame(dict(uid=[1,1,2], evt=['click','view','click'], amt=[10,20,30], ts=[100,200,300]))
src = DataSource(df, cats=['evt'], conts=['amt'], time_col='ts', entity_col='uid')
tok = Tokenizer()
res = tok._melt_events(src).collect()

test_eq(res.shape, (6,6))
test_eq(set(res.columns), {'uid','key','value','vtype','time','event_idx'})
test_eq(sorted(res.filter(pl.col('uid')==1)['event_idx'].unique().to_list()), [0,1])
test_eq(res.filter(pl.col('uid')==2)['event_idx'].unique().to_list(), [0])
test_eq(sorted(res['time'].unique().to_list()), [100,200,300])
test_eq(res.filter(pl.col('key')=='evt')['vtype'].unique().to_list(), ['cat'])
test_eq(res.filter(pl.col('key')=='amt')['vtype'].unique().to_list(), ['num'])

In [ ]:
df = pl.LazyFrame(dict(
    uid=[1,1],
    evt=['login','logout'],
    amt=[5,10],
    ts=[datetime(2020,6,15,12,0,0,tzinfo=timezone.utc), datetime(2020,6,15,12,30,0,tzinfo=timezone.utc)]))
src = DataSource(df, cats=['evt'], conts=['amt'], time_col='ts', entity_col='uid')
tok = Tokenizer()
res = tok._melt_events(src).collect()
exp = pl.Series(df.select('ts').collect()['ts']).dt.epoch(time_unit='s').to_list()

test_eq(sorted(res['time'].unique().to_list()), exp)
test_eq(len(res['time'].unique()), 2)

In [ ]:
#| export
@patch
def _melt_lifelong(self:Tokenizer, src:DataSource):
    if not src.lifelong: return pl.LazyFrame(schema=[src.entity_col, 'key', 'value', 'vtype', 'time'])

    parts = []
    for lc in src.lifelong:
        parts.append(
            src.df
            .select(src.entity_col, lc)
            .filter(pl.col(lc).is_not_null())
            .with_columns(
                pl.lit('lifelong').alias('key'),
                pl.lit(lc).alias('value'),
                pl.lit('cat').alias('vtype'),
                pl.col(lc).dt.epoch(time_unit='s').cast(pl.Int64).alias('time'),
            )
            .drop(lc)
        )

    return pl.concat(parts)

In [ ]:
df = pl.LazyFrame(dict(
    uid=[1,2],
    first_topup=[datetime(2020,1,1), datetime(2021,6,15)],
    first_trade=[datetime(2020,3,10), None],
    extra=[99,100]))
src = DataSource(df, conts=['extra'], lifelong=['first_topup','first_trade'], entity_col='uid')
tok = Tokenizer()
res = tok._melt_lifelong(src).collect()

test_eq(res.shape, (3,5))
test_eq(set(res.columns), {'uid','key','value','vtype','time'})
test_eq(res['key'].unique().to_list(), ['lifelong'])
test_eq(res['vtype'].unique().to_list(), ['cat'])
test_eq(set(res['value'].unique().to_list()), {'first_topup','first_trade'})
test_eq(res.filter(pl.col('uid')==1, pl.col('value')=='first_topup')['time'].item(), df.select(pl.col('first_topup').first().dt.epoch(time_unit='s')).collect().item())

src = DataSource(df, lifelong=[], entity_col='uid')
test_eq(tok._melt_lifelong(src).collect().height, 0)

In [ ]:
#| export
@patch
def _split_signed_nums(self:Tokenizer, kvt, signed_nums):
    signed_nums = list(signed_nums)
    signed = kvt.filter(pl.col('key').is_in(signed_nums), pl.col('vtype') == 'num').with_columns(pl.col('value').cast(pl.Float64, strict=False).alias('_fval'))
    mag = signed.with_columns(pl.col('_fval').abs().cast(pl.Utf8).alias('value')).drop('_fval')
    sign = signed.with_columns(
        (pl.col('key') + '_sign').alias('key'),
        pl.when(pl.col('_fval').is_null()).then(pl.lit('unknown')).when(pl.col('_fval') < 0).then(pl.lit('negative')).when(pl.col('_fval') > 0).then(pl.lit('positive')).otherwise(pl.lit('zero')).alias('value'),
        pl.lit('cat').alias('vtype')
    ).drop('_fval')
    rest = kvt.filter(~(pl.col('key').is_in(signed_nums) & (pl.col('vtype') == 'num')))
    return pl.concat([rest, mag, sign], how='diagonal_relaxed')

In [ ]:
#| export
@patch
def _to_kvt(self:Tokenizer, src:DataSource):
    regular = self._melt_profile_fields(src) if src.is_profile else self._melt_events(src)
    lifelong = self._melt_lifelong(src)
    parts = [p for p in [regular, lifelong] if len(p.collect_schema()) > 0]
    if not parts: return pl.LazyFrame(schema=[src.entity_col, 'key', 'value', 'vtype', 'time'])
    res = pl.concat(parts, how='diagonal_relaxed')
    res = self._split_signed_nums(res, src.signed_conts)
    res = res.with_columns(pl.col('value').cast(pl.Utf8).fill_null(''), pl.col('time').fill_null(0))
    keep = [c for c in [src.entity_col, 'key', 'value', 'vtype', 'time', 'event_idx'] if c in res.collect_schema().names()]
    return res.select(keep)

In [ ]:
df = pl.LazyFrame(dict(
    uid=list(range(1,4)),
    gender=['M','F','I'],
    age=[25,30,-30],
    first_topup=[datetime(2020,1,1), datetime(2021,6,15), datetime(2021,6,15)],
    first_trade=[datetime(2020,3,10), None, datetime(2021,6,15)]))
src = DataSource(df, cats=['gender'], conts=['age'], signed_conts=['age'], lifelong=['first_topup','first_trade'], entity_col='uid', is_profile=True)
tok = Tokenizer()
res = tok._to_kvt(src).collect()

test_eq(res.shape, (14,5))
test_eq(set(res.columns), {'uid','key','value','vtype','time'})
test_eq(set(res.filter(pl.col('key')=='age_sign')['value'].to_list()), {'positive','negative'})
test_eq(res.filter(pl.col('key')=='age')['value'].cast(pl.Float64).abs().to_list(), [25.,30.,30.])
test_eq(res.filter(pl.col('value').is_in(['first_topup','first_trade']))['time'].min() > 0, True)
test_eq(res.filter(pl.col('uid')==2, pl.col('value')=='first_trade').height, 0)

In [ ]:
res

uid,key,value,vtype,time
i64,str,str,str,i64
1,"""gender""","""M""","""cat""",0
2,"""gender""","""F""","""cat""",0
3,"""gender""","""I""","""cat""",0
1,"""lifelong""","""first_topup""","""cat""",1577836800
2,"""lifelong""","""first_topup""","""cat""",1623715200
…,…,…,…,…
2,"""age""","""30.0""","""num""",0
3,"""age""","""30.0""","""num""",0
1,"""age_sign""","""positive""","""cat""",0


## Tokenizer

Builds key and value vocabularies, bucketizes numerical values, and tokenizes source schemas into fixed-size tensors for the model.

- Each key gets one token in the key vocab
- Categorical values: one token per unique value
- Numerical values: percentile-bucket index
- Textual values: BPE subword tokens

Keys use a single shared vocabulary across all sources. `replace_strict` maps known keys to their index; any unknown key gets the `[UNK]` fallback.

In [ ]:
#| export
@patch
def _map_keys(self:Tokenizer, kvt):
    return kvt.with_columns(pl.col('key').replace_strict(self.key_vocab, default=self.key_vocab['[UNK]']).alias('key_id'))

In [ ]:
# Test: known keys → correct IDs, unknown key → [UNK]
tok = Tokenizer()
tok._add_key('gender'); tok._add_key('age')

kvt = pl.LazyFrame({
    'key': ['gender', 'age', 'unknown'],
    'value': ['M', '25', 'x'],
    'vtype': ['cat', 'num', 'text'],
    'time': [0, 0, 0]
})
result = tok._map_keys(kvt).collect()
test_eq(result['key_id'].to_list(), [tok.key_vocab['gender'], tok.key_vocab['age'], tok.key_vocab['[UNK]']])

In [ ]:
tok.key_vocab.items(), result

(dict_items([('[PAD]', 0), ('[MASK]', 1), ('[UNK]', 2), ('[USR]', 3), ('[EVT]', 4), ('gender', 5), ('age', 6)]),
 shape: (3, 5)
 ┌─────────┬───────┬───────┬──────┬────────┐
 │ key     ┆ value ┆ vtype ┆ time ┆ key_id │
 │ ---     ┆ ---   ┆ ---   ┆ ---  ┆ ---    │
 │ str     ┆ str   ┆ str   ┆ i64  ┆ i64    │
 ╞═════════╪═══════╪═══════╪══════╪════════╡
 │ gender  ┆ M     ┆ cat   ┆ 0    ┆ 5      │
 │ age     ┆ 25    ┆ num   ┆ 0    ┆ 6      │
 │ unknown ┆ x     ┆ text  ┆ 0    ┆ 2      │
 └─────────┴───────┴───────┴──────┴────────┘)

Categorical values map one-to-one to vocabulary tokens. PRAGMA uses this for high-frequency fields like MCC codes — one value, one token, direct lookup.

In [ ]:
#| export
@patch
def _map_cat_vals(self:Tokenizer, kvt):
    return (
        kvt.filter(pl.col('vtype') == 'cat')
        .with_columns(
            pl.col('value').replace_strict(self.val_vocab, default=self.val_vocab['[UNK]']).alias('val_id'),
            pl.lit(0).cast(pl.Int64).alias('val_pos')
        )
    )

In [ ]:
tok = Tokenizer()
tok._add_val('M'); tok._add_val('F')

kvt = pl.LazyFrame({
    'key': ['gender', 'gender', 'gender', 'age'],
    'value': ['M', 'unknown', 'F', '25'],
    'vtype': ['cat', 'cat', 'cat', 'num'],
    'time': [0, 0, 0, 1]
})
result = tok._map_cat_vals(kvt).collect().sort('value')
test_eq(result['val_id'].to_list() ,[ tok.val_vocab['F'], tok.val_vocab['M'], tok.val_vocab['[UNK]'],])

In [ ]:
tok.val_vocab.items(), result

(dict_items([('[PAD]', 0), ('[MASK]', 1), ('[UNK]', 2), ('M', 3), ('F', 4)]),
 shape: (3, 6)
 ┌────────┬─────────┬───────┬──────┬────────┬─────────┐
 │ key    ┆ value   ┆ vtype ┆ time ┆ val_id ┆ val_pos │
 │ ---    ┆ ---     ┆ ---   ┆ ---  ┆ ---    ┆ ---     │
 │ str    ┆ str     ┆ str   ┆ i64  ┆ i64    ┆ i64     │
 ╞════════╪═════════╪═══════╪══════╪════════╪═════════╡
 │ gender ┆ F       ┆ cat   ┆ 0    ┆ 4      ┆ 0       │
 │ gender ┆ M       ┆ cat   ┆ 0    ┆ 3      ┆ 0       │
 │ gender ┆ unknown ┆ cat   ┆ 0    ┆ 2      ┆ 0       │
 └────────┴─────────┴───────┴──────┴────────┴─────────┘)

Continuous values are discretized into percentile bins learned per key during `fit()`.

- `0` maps to the special numerical bucket `<bucket_0>`
- null / missing / unparsable values map to `[UNK]`
- non-zero numeric values map to percentile buckets `<bucket_1>`, `<bucket_2>`, ...
- `np.searchsorted` finds the bucket index from the precomputed boundaries

This keeps all model inputs discrete while preserving approximate magnitude/order information.

In [ ]:
#| export
@patch
def _map_num_vals(self:Tokenizer, kvt):
    nums = kvt.filter(pl.col('vtype') == 'num')
    unk = lambda df: df.with_columns(pl.lit(self.val_vocab['[UNK]']).alias('val_id'), pl.lit(0).cast(pl.Int64).alias('val_pos'))
    if not self.num_bounds: return unk(nums)

    def enc_key(k,bounds):
        return (
            nums.filter(pl.col('key') == k)
            .with_columns(pl.col('value').cast(pl.Float64, strict=False).alias('_fval'))
            .with_columns(
                pl.when(pl.col('_fval').is_null()).then(None)
                .when(pl.col('_fval') == 0).then(0)
                .otherwise(pl.col('_fval').map_batches(lambda s: np.searchsorted(bounds, s.to_numpy())+1, return_dtype=pl.Int64)).alias('_bucket_idx'))
            .with_columns(
                pl.when(pl.col('_bucket_idx').is_null()).then(pl.lit(self.val_vocab['[UNK]']))
                .otherwise(pl.format('<bucket_{}>', '_bucket_idx').replace_strict(self.val_vocab, default=self.val_vocab['[UNK]'])).alias('val_id'),
                pl.lit(0).cast(pl.Int64).alias('val_pos'))
            .drop('_fval', '_bucket_idx')
        )

    parts = [enc_key(k,bounds) for k,bounds in self.num_bounds.items()]
    parts.append(unk(nums.filter(~pl.col('key').is_in(list(self.num_bounds)))))
    return pl.concat(parts, how='diagonal_relaxed')

In [ ]:
tok = Tokenizer(num_buckets=4)
for i in range(tok.num_buckets+1): tok._add_val(f'<bucket_{i}>')
tok.num_bounds['amt'] = np.array([2.5, 5.0, 7.5])

kvt = pl.LazyFrame(dict(
    key=['amt','amt','amt','amt','amt','amt','val'],
    value=[None,'0.0','2.5','5.0','-3.0','999.0','9'],
    vtype=['num']*7,
    time=[0]*7))

result = tok._map_num_vals(kvt).collect()

want = {
    None: '[UNK]',
    '0.0': '<bucket_0>',
    '2.5': '<bucket_1>',
    '5.0': '<bucket_2>',
    '-3.0': '<bucket_1>',
    '999.0': '<bucket_4>',
}

for v,b in want.items(): test_eq(result.filter(pl.col('key')=='amt', pl.col('value').eq_missing(v))['val_id'].item(), tok.val_vocab[b])
test_eq(result.filter(pl.col('key')=='val')['val_id'].item(), tok.val_vocab['[UNK]'])

In [ ]:
result

key,value,vtype,time,val_id,val_pos
str,str,str,i64,i64,i64
"""amt""",null,"""num""",0,2,0
"""amt""","""0.0""","""num""",0,3,0
"""amt""","""2.5""","""num""",0,4,0
"""amt""","""5.0""","""num""",0,5,0
"""amt""","""-3.0""","""num""",0,4,0
"""amt""","""999.0""","""num""",0,7,0
"""val""","""9""","""num""",0,2,0


High-cardinality text gets BPE subword tokenization via tiktoken. Each subword becomes its own row sharing the same key — the model groups them later via shared key embeddings and within-field positional encodings.

Using a dynamic `bpe_offset = len(val_vocab)` computed at encode time means BPE IDs naturally shift as `val_vocab` grows — add a new categorical token, and all subword IDs move up by one. That's fine as long as you re-encode the entire dataset whenever you refit. Unknown keys/values/new numeric fields all route to `[UNK]` gracefully, so a new source won't error — it'll just need refitting (and re-encoding) to learn from it properly.

In [ ]:
#| export
@patch
def _encode_text(self:Tokenizer, v):
    ids = self.bpe_tokenizer.encode(v).ids
    if not ids: return [self.val_vocab['[UNK]']]
    return [self.bpe_offset+i for i in ids]

In [ ]:
#| export
@patch
def _norm_text(self:Tokenizer, v): return '[UNK]' if v is None or v == '' else v

In [ ]:
#| export
@patch
def _map_text_vals(self:Tokenizer, kvt):
    lowcard_keys = getattr(self, 'lowcard_text_keys', set())
    txt = kvt.filter(pl.col('vtype') == 'text').with_columns(pl.col('value').map_elements(self._norm_text, return_dtype=pl.Utf8).alias('value'))
    low = txt.filter(pl.col('key').is_in(lowcard_keys)).with_columns(pl.col('value').replace_strict(self.val_vocab, default=self.val_vocab['[UNK]']).alias('val_id'), pl.lit(0).cast(pl.Int64).alias('val_pos'))
    high = txt.filter(~pl.col('key').is_in(lowcard_keys))
    if self.bpe_tokenizer is None and high.limit(1).collect().height: raise ValueError("high-cardinality text present but no BPE tokenizer was fitted")
    if self.bpe_tokenizer is None: return low
    high = high.with_columns(pl.col('value').map_elements(lambda v: self._encode_text(v), return_dtype=pl.List(pl.Int64)).alias('val_ids')).with_columns(pl.int_ranges(0, pl.col('val_ids').list.len()).alias('val_pos')).explode('val_ids', 'val_pos', empty_as_null=True).rename({'val_ids': 'val_id'})
    return pl.concat([low, high], how='diagonal_relaxed')

In [ ]:
tok = Tokenizer()
tok._add_key('desc'); tok._add_key('desc_short'); tok._add_val('hello')
tok.lowcard_text_keys.add('desc_short')

class MockEncoded:
    def __init__(self, ids): self.ids = ids

class MockTok:
    def encode(self, text): return MockEncoded([7,8,7] if 'plan' in text else [9])

tok.bpe_tokenizer = MockTok()
off = 0

kvt = pl.LazyFrame(dict(
    key=['desc', 'desc', 'desc_short'],
    value=['metal plan', 'world', 'hello'],
    vtype=['text']*3,
    time=[0]*3))

res = tok._map_text_vals(kvt).collect()
mp = res.filter(pl.col('key')=='desc', pl.col('value')=='metal plan')
w = res.filter(pl.col('key')=='desc', pl.col('value')=='world')
ds = res.filter(pl.col('key')=='desc_short')

test_eq(mp['val_id'].to_list(), [off+7, off+8, off+7])
test_eq(mp['val_pos'].to_list(), [0,1,2])
test_eq(w['val_id'].item(), off+9)
test_eq(w['val_pos'].item(), 0)
test_eq(ds['val_id'].item(), tok.val_vocab['hello'])
test_eq(ds['val_pos'].item(), 0)
test_eq(res.height, 5)

In [ ]:
result

key,value,vtype,time,val_id,val_pos
str,str,str,i64,i64,i64
"""amt""",null,"""num""",0,2,0
"""amt""","""0.0""","""num""",0,3,0
"""amt""","""2.5""","""num""",0,4,0
"""amt""","""5.0""","""num""",0,5,0
"""amt""","""-3.0""","""num""",0,4,0
"""amt""","""999.0""","""num""",0,7,0
"""val""","""9""","""num""",0,2,0


A single-pass scan builds all vocabularies and parameters: key vocab across all sources, val vocab from categoricals, percentile boundaries for continuous columns, and identifies low-cardinality text fields to treat as categorical.

In [ ]:
#| export
@patch
def fit(self:Tokenizer, sources:L, eval_points:pl.LazyFrame=None):
    sources = L(sources)
    kvt = pl.concat(sources.map(self._to_kvt), how='diagonal_relaxed').collect()

    for k in sorted(kvt['key'].unique()): self._add_key(k)

    cat_vals = kvt.filter(pl.col('vtype') == 'cat')['value'].unique()
    for v in sorted(cat_vals): self._add_val(v)

    num_df = kvt.filter(pl.col('vtype') == 'num')
    qs = np.linspace(0, 100, self.num_buckets+1)[1:-1]
    for k in sorted(num_df['key'].unique()):
        vals = num_df.filter(pl.col('key') == k)['value'].cast(pl.Float64, strict=False).drop_nulls()
        vals = vals.filter(vals != 0)
        if len(vals): self.num_bounds[k] = np.percentile(vals.to_numpy(), qs)

    for i in range(self.num_buckets+1): self._add_val(f'<bucket_{i}>')

    text_df = kvt.filter(pl.col('vtype') == 'text')
    high_texts = []
    for k in sorted(text_df['key'].unique()):
        vals = text_df.filter(pl.col('key') == k)['value']
        if vals.n_unique() <= self.cardinality_threshold:
            self.lowcard_text_keys.add(k)
            for v in sorted(vals.unique()): self._add_val(v)
        else: high_texts += vals.to_list()

    self.bpe_offset = len(self.val_vocab)
    if high_texts:
        self.bpe_tokenizer = HFTokenizer(models.BPE(unk_token='[UNK]'))
        self.bpe_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
        trainer = trainers.BpeTrainer(vocab_size=28000, special_tokens=['[UNK]', '[PAD]', '[MASK]'], min_frequency=2)
        self.bpe_tokenizer.train_from_iterator(high_texts, trainer)
    else: self.bpe_tokenizer = None

    return self

In [ ]:
#| test
tok = Tokenizer(num_buckets=5)
df = pl.LazyFrame(dict(uid=[1,2], gender=['M','F'], age=[25,30], first_topup=[datetime(2020,1,1), datetime(2021,6,15)]))
src = DataSource(df, cats=['gender'], conts=['age'], lifelong=['first_topup'], entity_col='uid', is_profile=True)
tok.fit(L([src]))

test_eq('gender' in tok.key_vocab, True)
test_eq('age' in tok.key_vocab, True)
test_eq('lifelong' in tok.key_vocab, True)
test_eq('first_topup' in tok.val_vocab, True)
test_eq('M' in tok.val_vocab, True)
test_eq('F' in tok.val_vocab, True)
test_eq('age' in tok.num_bounds, True)
test_eq(len(tok.num_bounds['age']), tok.num_buckets-1)
test_eq([f'<bucket_{i}>' in tok.val_vocab for i in range(tok.num_buckets+1)], [True]*(tok.num_buckets+1))

kvt = tok._to_kvt(src)
res = tok._map_keys(kvt).collect()
test_eq(set(res.columns), {'uid','key','value','vtype','time','key_id'})
test_eq(res.filter(pl.col('key')=='gender')['key_id'].unique().item(), tok.key_vocab['gender'])
test_eq(res.filter(pl.col('key')=='age')['key_id'].unique().item(), tok.key_vocab['age'])

res = tok._map_cat_vals(kvt).collect()
test_eq(set(res['value']), {'M','F','first_topup'})
test_eq((res['val_pos'] == 0).all(), True)

res = tok._map_num_vals(kvt).collect()
test_eq(res.height, 2)
test_eq((res['val_id'] != tok.val_vocab['[UNK]']).all(), True)
test_eq((res['val_pos'] == 0).all(), True)

tok = Tokenizer(num_buckets=5, cardinality_threshold=2)
df = pl.LazyFrame(dict(uid=[1,2,3], high_text=['the quick brown fox','hello world again','something else'], low_text=['x','y','x']))
src = DataSource(df, texts=['high_text','low_text'], entity_col='uid', is_profile=True)
tok.fit(L([src]))

test_eq('low_text' in tok.lowcard_text_keys, True)
test_eq('x' in tok.val_vocab and 'y' in tok.val_vocab, True)
test_eq('high_text' in tok.lowcard_text_keys, False)
test_eq(tok.bpe_tokenizer is not None, True)

txt = tok._map_text_vals(tok._to_kvt(src)).collect()
test_eq(txt.height >= 5, True)
test_eq((txt['val_id'] != tok.val_vocab['[UNK]']).all(), True)

df1 = pl.LazyFrame(dict(uid=[1], cat1=['a'], num1=[10]))
df2 = pl.LazyFrame(dict(uid=[2], cat2=['b'], num2=[20]))
src1 = DataSource(df1, cats=['cat1'], conts=['num1'], entity_col='uid', is_profile=True)
src2 = DataSource(df2, cats=['cat2'], conts=['num2'], entity_col='uid', is_profile=True)
tok = Tokenizer()
tok.fit(L([src1, src2]))

test_eq(all(o in tok.key_vocab for o in ['cat1','cat2','num1','num2']), True)
test_eq(all(o in tok.val_vocab for o in ['a','b']), True)
test_eq(all(o in tok.num_bounds for o in ['num1','num2']), True)

n_keys,n_vals,n_bounds = len(tok.key_vocab),len(tok.val_vocab),len(tok.num_bounds)
tok.fit(L([src1, src2]))
test_eq((len(tok.key_vocab),len(tok.val_vocab),len(tok.num_bounds)), (n_keys,n_vals,n_bounds))

tok = Tokenizer()
src = DataSource(pl.LazyFrame(dict(uid=[1])), entity_col='uid', is_profile=True)
tok.fit(L([src]))
test_eq(tok.key_vocab, {'[PAD]':0, '[MASK]':1, '[UNK]':2, '[USR]':3, '[EVT]':4})
test_eq(tok.num_bounds, {})
test_eq(tok.bpe_tokenizer, None)

## Time Features

Tokenization keeps event timestamps as raw epoch seconds until this stage. Events use log-seconds to the latest event in the entity history; lifelong profile fields use log-seconds to the evaluation point; regular profile fields stay at zero.

In [ ]:
#| export
def _to_epoch(ts):
    if isinstance(ts, (int, float)): return int(ts)
    return pl.Series([str(ts)]).str.to_datetime().dt.epoch(time_unit='s').item()

In [ ]:
#| export
def _log_s(ts): return 8*np.log1p(np.maximum(ts, 0)/8) # ts in seconds expected

In [ ]:
#| export
@patch
def _map_values(self:Tokenizer, kvt):
    cats  = self._map_cat_vals(kvt)
    nums  = self._map_num_vals(kvt)
    texts = self._map_text_vals(kvt)
    return pl.concat([cats, nums, texts], how='diagonal_relaxed')

In [ ]:
#| export
@patch
def encode_source(self:Tokenizer, src:DataSource):
    kvt = self._map_keys(self._to_kvt(src))
    vals = self._map_values(kvt)
    keep = [c for c in [src.entity_col,'key','key_id','value','val_id','val_pos','vtype','time','event_idx'] if c in vals.collect_schema().names()]
    return vals.select(keep)

Truncation (max events per user, max tokens per event) belongs in the **dataloader**, not in tokenization. Tokenization produces the full flat parquet per source; the dataloader applies truncation at load time so it can be tuned per experiment without re-tokenizing.

In [ ]:
#| export
@patch
def _tokenize(self:Tokenizer, src:DataSource, eval_time=None, entity_max_ts=None):
    if eval_time is None: raise ValueError('eval_time must be provided')
    kvt = self.encode_source(src)
    names = kvt.collect_schema().names()

    if 'event_idx' in names:
        kvt = kvt.join(entity_max_ts, on=src.entity_col, how='left') if entity_max_ts is not None else kvt.with_columns(pl.col('time').max().over(src.entity_col).alias('max_epoch'))
        return kvt.with_columns(
            pl.col('max_epoch').sub(pl.col('time')).map_batches(lambda x: pl.Series(_log_s(x.to_numpy().astype(np.float64))), return_dtype=pl.Float64).alias('logsec'),
            pl.from_epoch('time', time_unit='s').dt.hour().cast(pl.Float32).alias('hour'),
            pl.from_epoch('time', time_unit='s').dt.weekday().cast(pl.Float32).alias('dow'),
            pl.from_epoch('time', time_unit='s').dt.day().cast(pl.Float32).alias('dom')
        ).drop('max_epoch')

    eval_epoch = _to_epoch(eval_time)
    return kvt.with_columns(
        pl.when(pl.col('time') > 0)
        .then(pl.col('time').map_batches(lambda x: pl.Series(_log_s((eval_epoch - x.to_numpy()).astype(np.float64))), return_dtype=pl.Float64))
        .otherwise(0.0).alias('logsec')
    )

In [ ]:
tok = Tokenizer(num_buckets=4)

pdf = pl.LazyFrame(dict(uid=[1,2], plan=['free','pro'], first_seen=[datetime(2020,1,1,tzinfo=timezone.utc), None]))
edf = pl.LazyFrame(dict(uid=[1,1,2], evt=['view','buy','view'], amt=[0,10,20], ts=[100,200,300]))

psrc = DataSource(pdf, cats=['plan'], lifelong=['first_seen'], entity_col='uid', is_profile=True)
esrc = DataSource(edf, cats=['evt'], conts=['amt'], time_col='ts', entity_col='uid')

tok.fit(L([psrc, esrc]))

prof = tok._tokenize(psrc, eval_time=datetime(2020,1,11,tzinfo=timezone.utc)).collect()
evts = tok._tokenize(esrc, eval_time=0).collect()

test_eq(set(['uid','key','key_id','value','val_id','val_pos','vtype','time','logsec']).issubset(prof.columns), True)
test_eq(set(['uid','key','key_id','value','val_id','val_pos','vtype','time','event_idx','logsec','hour','dow','dom']).issubset(evts.columns), True)

test_eq(prof.filter(pl.col('key')=='plan')['logsec'].to_list(), [0.0,0.0])
test_eq(prof.filter(pl.col('value')=='first_seen').height, 1)
test_eq(abs(prof.filter(pl.col('value')=='first_seen')['logsec'].item() - _log_s(10*24*60*60)) < 1e-6, True)

test_eq(evts.filter(pl.col('uid')==1, pl.col('time')==200)['logsec'].unique().item(), 0.0)
test_eq(abs(evts.filter(pl.col('uid')==1, pl.col('time')==100, pl.col('key')=='evt')['logsec'].item() - _log_s(100)) < 1e-6, True)
test_eq(evts.filter(pl.col('uid')==2)['logsec'].unique().to_list(), [0.0])
test_eq(evts.filter(pl.col('uid')==1)['event_idx'].unique().sort().to_list(), [0,1])

test_eq((prof['val_id'] != tok.val_vocab['[UNK]']).all(), True)
test_eq((evts.filter(pl.col('key')!='amt')['val_id'] != tok.val_vocab['[UNK]']).all(), True)

In [ ]:
profile = DataSource.from_file(path / "u.user", separator="|", has_header=False,
    new_columns=["user_id","age","gender","occupation","zip_code"], 
    cats=["gender","occupation"], conts=["age"], entity_col="user_id",
    is_profile=True)
events = DataSource.from_file(path / "u.data", separator="\t", has_header=False,
    new_columns=["user_id", "movie_id", "rating", "timestamp"],
    cats=["movie_id"], conts=["rating"], time_col="timestamp", entity_col="user_id")

In [ ]:
tok = Tokenizer(num_buckets=10, cardinality_threshold=100)
tok.fit(L([profile, events]))

profile_tok = tok._tokenize(profile, eval_time='1998-04-01T00:00:00').head(3).collect()
events_tok = tok._tokenize(events, eval_time='1998-04-01T00:00:00').head(2).collect()

print("Profile:", profile_tok)
print("Events:", events_tok)

Profile: shape: (3, 9)
┌─────────┬────────┬────────┬───────┬───┬─────────┬───────┬──────┬────────┐
│ user_id ┆ key    ┆ key_id ┆ value ┆ … ┆ val_pos ┆ vtype ┆ time ┆ logsec │
│ ---     ┆ ---    ┆ ---    ┆ ---   ┆   ┆ ---     ┆ ---   ┆ ---  ┆ ---    │
│ i64     ┆ str    ┆ i64    ┆ str   ┆   ┆ i64     ┆ str   ┆ i64  ┆ f64    │
╞═════════╪════════╪════════╪═══════╪═══╪═════════╪═══════╪══════╪════════╡
│ 1       ┆ gender ┆ 6      ┆ M     ┆ … ┆ 0       ┆ cat   ┆ 0    ┆ 0.0    │
│ 2       ┆ gender ┆ 6      ┆ F     ┆ … ┆ 0       ┆ cat   ┆ 0    ┆ 0.0    │
│ 3       ┆ gender ┆ 6      ┆ M     ┆ … ┆ 0       ┆ cat   ┆ 0    ┆ 0.0    │
└─────────┴────────┴────────┴───────┴───┴─────────┴───────┴──────┴────────┘


Events: shape: (2, 13)
┌─────────┬──────────┬────────┬───────┬───┬────────────┬──────┬─────┬──────┐
│ user_id ┆ key      ┆ key_id ┆ value ┆ … ┆ logsec     ┆ hour ┆ dow ┆ dom  │
│ ---     ┆ ---      ┆ ---    ┆ ---   ┆   ┆ ---        ┆ ---  ┆ --- ┆ ---  │
│ i64     ┆ str      ┆ i64    ┆ str   ┆   ┆ f64        ┆ f32  ┆ f32 ┆ f32  │
╞═════════╪══════════╪════════╪═══════╪═══╪════════════╪══════╪═════╪══════╡
│ 1       ┆ movie_id ┆ 7      ┆ 168   ┆ … ┆ 115.438142 ┆ 21.0 ┆ 1.0 ┆ 22.0 │
│ 1       ┆ movie_id ┆ 7      ┆ 172   ┆ … ┆ 115.438142 ┆ 21.0 ┆ 1.0 ┆ 22.0 │
└─────────┴──────────┴────────┴───────┴───┴────────────┴──────┴─────┴──────┘


## Tokenizer Persistence

The fitted tokenizer is part of the dataset contract. Save key/value vocabularies, numeric bucket boundaries, low-cardinality text choices, and any BPE model so tokenized train/valid/test data can be reproduced exactly.

In [ ]:
#| export
@patch
def save(self:Tokenizer, path='tokenizer.json'):
    path = Path(path)
    state = dict(key_vocab=dict(self.key_vocab), val_vocab=dict(self.val_vocab), num_bounds={k:v.tolist() for k,v in self.num_bounds.items()},
        lowcard_text_keys=list(self.lowcard_text_keys), num_buckets=self.num_buckets, cardinality_threshold=self.cardinality_threshold)
    if self.bpe_tokenizer is not None:
        bpe_path = path.with_name(f'{path.stem}_bpe{path.suffix}')
        self.bpe_tokenizer.save(str(bpe_path))
        state['bpe_path'] = bpe_path.name
    path.write_text(json.dumps(state, indent=2))
    return path

In [ ]:
#| export
@patch(cls_method=True)
def load(cls:Tokenizer, path='tokenizer.json'):
    path = Path(path)
    state = json.loads(path.read_text())
    tok = cls(num_buckets=state['num_buckets'], cardinality_threshold=state['cardinality_threshold'])
    tok.key_vocab = state['key_vocab']
    tok.val_vocab = state['val_vocab']
    tok.num_bounds = {k:np.array(v) for k,v in state['num_bounds'].items()}
    tok.lowcard_text_keys = set(state['lowcard_text_keys'])
    tok.bpe_offset = len(tok.val_vocab)
    tok.bpe_tokenizer = HFTokenizer.from_file(str(path.with_name(state['bpe_path']))) if 'bpe_path' in state else None
    return tok

In [ ]:
tmp = Path(tempfile.mkdtemp())/'tokenizer.json'
tok.save(tmp)
tok2 = Tokenizer.load(tmp)
test_eq(tok2.key_vocab, tok.key_vocab)
test_eq(tok2.val_vocab, tok.val_vocab)
test_eq(set(tok2.num_bounds), set(tok.num_bounds))
test_eq(tok2.bpe_offset, len(tok2.val_vocab))

## PRAGMADataset

Assemble profile + event sources into per-entity tokenized sequences. Fits a tokenizer across all sources and produces tokenized tensors keyed by entity.

Usage:
    ds = PRAGMADataset(static=profile_schema, dynamic=[events_schema], entity_col='user_id')
    batches = ds.tokenize()
    ds.show_batch(1)

## Shared Event Time Reference

When multiple event sources exist, each entity needs one global latest timestamp across all event sources. Otherwise every source would incorrectly treat its own latest event as `logsec=0`.

In [ ]:
#| export
def _compute_entity_max_ts(sources):
    "Return a LazyFrame with entity_col and max_epoch across all event sources."
    parts,entity_col = [],None
    for src in sources:
        if src.is_profile or not src.time_col: continue
        if entity_col is None: entity_col = src.entity_col
        assert src.entity_col == entity_col
        typ = src.df.collect_schema()[src.time_col]
        time = pl.col(src.time_col).dt.epoch(time_unit='s').cast(pl.Int64) if typ.is_temporal() else pl.col(src.time_col).cast(pl.Int64)
        parts.append(src.df.select(src.entity_col, src.time_col).with_columns(time.alias('epoch')).drop(src.time_col))
    if not parts: return None
    return pl.concat(parts).group_by(entity_col).agg(pl.col('epoch').max().alias('max_epoch'))

In [ ]:
df1 = pl.LazyFrame(dict(uid=[1,1,2], ts=[100,300,200]))
df2 = pl.LazyFrame(dict(uid=[1,2], ts=[500,50]))
prof = DataSource(pl.LazyFrame(dict(uid=[1])), entity_col='uid', is_profile=True)
src1,src2 = [DataSource(o, time_col='ts', entity_col='uid') for o in (df1,df2)]

res = _compute_entity_max_ts([src1]).collect().sort('uid')
test_eq(res['uid'].to_list(), [1,2])
test_eq(res['max_epoch'].to_list(), [300,200])
test_eq(_compute_entity_max_ts([prof]), None)

res = _compute_entity_max_ts([prof, src1, src2]).collect().sort('uid')
test_eq(res['uid'].to_list(), [1,2])
test_eq(res['max_epoch'].to_list(), [500,200])

## Dataset Orchestration

`PRAGMADataset` owns source consistency, tokenizer fitting, and sharded parquet writing. It preserves the caller's `entity_col` name and keeps truncation out of tokenization so dataloader experiments can change context limits without re-tokenizing.

In [ ]:
#| export
class PRAGMADataset:
    def __init__(self, profile=None, events=None, entity_col='entity_id', out_path=None):
        store_attr('profile,entity_col')
        self.events,self.tokenizer = L(events or []),None
        self.out_path = Path(ifnone(out_path, './data'))
        os.makedirs(self.out_path, exist_ok=True)
        self._check_sources()

    def _check_sources(self):
        srcs = ([self.profile] if self.profile is not None else []) + list(self.events)
        for i,s in enumerate(srcs):
            if s.entity_col != self.entity_col: raise ValueError(f"source {i} entity_col={s.entity_col!r} != {self.entity_col!r}")

    @property
    def sources(self): return L(([self.profile] if self.profile is not None else []) + list(self.events))

    def fit_tokenizer(self, num_buckets=100, cardinality_threshold=100):
        self.tokenizer = Tokenizer(num_buckets=num_buckets, cardinality_threshold=cardinality_threshold).fit(self.sources)
        tok = self.tokenizer
        print(f"Keys: {len(tok.key_vocab)}, Vals: {len(tok.val_vocab)}, BPE: {tok.bpe_tokenizer.get_vocab_size() if tok.bpe_tokenizer else 'none'}")
        tok.save(self.out_path/'tokenizer.json')
        return tok

Sharded this way because light on memory

In [ ]:
#| export
@patch
def tokenize(self:PRAGMADataset, eval_time, n_shards=100):
    "Tokenize sources to entity-sharded parquet files."
    if self.tokenizer is None: self.fit_tokenizer()
    entity_max_ts = _compute_entity_max_ts(self.events)
    parts = []

    if self.profile is not None:
        print('tokenizing profile')
        parts.append(self.tokenizer._tokenize(self.profile, eval_time=eval_time).with_columns(pl.lit(-1).alias('source_idx'), pl.lit(-1).alias('event_idx')))

    for i,src in enumerate(progress_bar(self.events)):
        print(f'tokenizing event source {i}: {src.name}')
        parts.append(self.tokenizer._tokenize(src, eval_time=eval_time, entity_max_ts=entity_max_ts).with_columns(pl.lit(i).alias('source_idx')))

    print('combining sources')
    lf = pl.concat(parts, how='diagonal_relaxed').with_columns(
        pl.col(self.entity_col).hash().mod(n_shards).alias('shard_id'),
        pl.col('hour').cast(pl.Float32).fill_null(0),
        pl.col('dow').cast(pl.Float32).fill_null(0),
        pl.col('dom').cast(pl.Float32).fill_null(0)
    ).sort(self.entity_col, 'source_idx', 'event_idx', 'key', 'val_pos')

    for s in progress_bar(range(n_shards)):
        lf.filter(pl.col('shard_id') == s).sink_parquet(self.out_path/f'shard_{s}.parquet')

    return self.out_path

In [ ]:
tmp = Path(tempfile.mkdtemp())
ds = PRAGMADataset(profile=psrc, events=[esrc], entity_col='uid', out_path=tmp)
tok = ds.fit_tokenizer(num_buckets=4)

test_eq(tok is ds.tokenizer, True)
test_eq((tmp/'tokenizer.json').exists(), True)
test_eq('plan' in tok.key_vocab, True)
test_eq('evt' in tok.key_vocab, True)
test_eq('amt' in tok.key_vocab, True)

Keys: 9, Vals: 13, BPE: none


In [ ]:
out = ds.tokenize(eval_time=datetime(2020,1,11,tzinfo=timezone.utc), n_shards=2)

test_eq(out, tmp)
test_eq(sorted(o.name for o in tmp.glob('*.parquet')), ['shard_0.parquet', 'shard_1.parquet'])

tokenizing profile


 |----------------------------------------| 0.00% [0/1 00:00<?]

tokenizing event source 0: None


 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

combining sources


 |----------------------------------------| 0.00% [0/2 00:00<?]

 |████████████████████--------------------| 50.00% [1/2 00:00<00:00]

 |████████████████████████████████████████| 100.00% [2/2 00:00<00:00]

In [ ]:
res = pl.concat([pl.scan_parquet(o) for o in tmp.glob('shard_*.parquet')]).collect()
prof = res.filter(pl.col('source_idx') == -1)
evts = res.filter(pl.col('source_idx') == 0)

test_eq(set(['uid','key','key_id','value','val_id','val_pos','vtype','time','logsec','source_idx','event_idx','shard_id']).issubset(prof.columns), True)
test_eq(set(['uid','key','key_id','value','val_id','val_pos','vtype','time','event_idx','logsec','hour','dow','dom','source_idx','shard_id']).issubset(evts.columns), True)

test_eq(prof['source_idx'].unique().to_list(), [-1])
test_eq(prof['event_idx'].unique().to_list(), [-1])
test_eq(evts['source_idx'].unique().to_list(), [0])
test_eq(evts.filter(pl.col('uid')==1, pl.col('time')==200)['logsec'].unique().item(), _log_s(0))
test_eq(abs(evts.filter(pl.col('uid')==1, pl.col('time')==100, pl.col('key')=='evt')['logsec'].item() - _log_s(100)) < 1e-6, True)

In [ ]:
bad_src = DataSource(pl.LazyFrame(dict(user_id=[1], evt=['x'], ts=[1])), cats=['evt'], time_col='ts', entity_col='user_id')
test_fail(lambda: PRAGMADataset(profile=psrc, events=[bad_src], entity_col='uid'))

In [ ]:
#| export
@patch
def show_summary(self:PRAGMADataset):
    rows = []
    for i,src in enumerate(self.sources):
        df = src.df
        schema = df.collect_schema()
        n = df.select(pl.len()).collect().item()
        ents = df.select(pl.col(src.entity_col).n_unique()).collect().item()
        row = dict(i=i, name=src.name, kind='profile' if src.is_profile else 'event', rows=n, entities=ents, entity_col=src.entity_col, time_col=src.time_col, cats=len(src.cats), conts=len(src.conts), signed_conts=len(src.signed_conts), texts=len(src.texts), lifelong=len(src.lifelong))
        if src.time_col:
            typ = schema[src.time_col]
            t = pl.col(src.time_col).dt.epoch(time_unit='s') if typ.is_temporal() else pl.col(src.time_col)
            mn,mx = df.select(t.min().alias('min_time'), t.max().alias('max_time')).collect().row(0)
            row = dict(row, min_time=mn, max_time=mx)
        rows.append(row)
    res = pl.DataFrame(rows)
    if self.tokenizer is None: return res
    tok = self.tokenizer
    print(f"sources: {len(self.sources)}")
    print(f"key vocab: {len(tok.key_vocab)}")
    print(f"val vocab: {len(tok.val_vocab)}")
    print(f"num fields: {len(tok.num_bounds)}")
    print(f"low-card text fields: {len(tok.lowcard_text_keys)}")
    print(f"bpe vocab: {tok.bpe_tokenizer.get_vocab_size() if tok.bpe_tokenizer else 'none'}")
    return res

In [ ]:
out = Path(tempfile.mkdtemp())/'ml100k_tok'
ds = PRAGMADataset(profile=profile, events=[events], entity_col='user_id', out_path=out)
ds.show_summary()

i,name,kind,rows,entities,entity_col,time_col,cats,conts,signed_conts,texts,lifelong,min_time,max_time
i64,str,str,i64,i64,str,str,i64,i64,i64,i64,i64,i64,i64
0,"""u""","""profile""",943,943,"""user_id""",null,2,1,0,0,0,null,null
1,"""u""","""event""",100000,943,"""user_id""","""timestamp""",1,1,0,0,0,874724710,893286638


## End-to-End Tokenization Check

MovieLens 100k is used as a compact integration test: fit the shared tokenizer, write sharded parquet, then verify row counts, token ids, profile/event conventions, and time features.

In [ ]:
out = Path(tempfile.mkdtemp())/'ml100k_tok'
ds = PRAGMADataset(profile=profile, events=[events], entity_col='user_id', out_path=out)
print(ds.show_summary())
tok = ds.fit_tokenizer(num_buckets=10, cardinality_threshold=100)
ds.tokenize(eval_time='1998-04-01T00:00:00', n_shards=4)
sorted(o.name for o in out.iterdir())

shape: (2, 14)
┌─────┬──────┬─────────┬────────┬───┬───────┬──────────┬───────────┬───────────┐
│ i   ┆ name ┆ kind    ┆ rows   ┆ … ┆ texts ┆ lifelong ┆ min_time  ┆ max_time  │
│ --- ┆ ---  ┆ ---     ┆ ---    ┆   ┆ ---   ┆ ---      ┆ ---       ┆ ---       │
│ i64 ┆ str  ┆ str     ┆ i64    ┆   ┆ i64   ┆ i64      ┆ i64       ┆ i64       │
╞═════╪══════╪═════════╪════════╪═══╪═══════╪══════════╪═══════════╪═══════════╡
│ 0   ┆ u    ┆ profile ┆ 943    ┆ … ┆ 0     ┆ 0        ┆ null      ┆ null      │
│ 1   ┆ u    ┆ event   ┆ 100000 ┆ … ┆ 0     ┆ 0        ┆ 874724710 ┆ 893286638 │
└─────┴──────┴─────────┴────────┴───┴───────┴──────────┴───────────┴───────────┘


Keys: 10, Vals: 1719, BPE: none


tokenizing profile


 |----------------------------------------| 0.00% [0/1 00:00<?]

tokenizing event source 0: u


...

 |██████████------------------------------| 25.00% [1/4 00:00<00:02]

 |████████████████████--------------------| 50.00% [2/4 00:01<00:01]

 |██████████████████████████████----------| 75.00% [3/4 00:02<00:00]

 |████████████████████████████████████████| 100.00% [4/4 00:02<00:00]

['shard_0.parquet',
 'shard_1.parquet',
 'shard_2.parquet',
 'shard_3.parquet',
 'tokenizer.json']

In [ ]:
len(tok.key_vocab), len(tok.val_vocab), tok.num_bounds.keys(), tok.bpe_tokenizer

(10, 1719, dict_keys(['age', 'rating']), None)

In [ ]:
toks = pl.concat([pl.scan_parquet(out/f'shard_{i}.parquet') for i in range(4)]).collect()
prof = toks.filter(pl.col('source_idx') == -1)
evts = toks.filter(pl.col('source_idx') == 0)
prof.shape, evts.shape, prof.columns, evts.columns

((2829, 15),
 (200000, 15),
 ['user_id',
  'key',
  'key_id',
  'value',
  'val_id',
  'val_pos',
  'vtype',
  'time',
  'logsec',
  'source_idx',
  'event_idx',
  'hour',
  'dow',
  'dom',
  'shard_id'],
 ['user_id',
  'key',
  'key_id',
  'value',
  'val_id',
  'val_pos',
  'vtype',
  'time',
  'logsec',
  'source_idx',
  'event_idx',
  'hour',
  'dow',
  'dom',
  'shard_id'])

In [ ]:
test_eq(prof.height, 943*3)
test_eq(evts.height, 100000*2)
test_eq(prof['source_idx'].unique().to_list(), [-1])
test_eq(prof['event_idx'].unique().to_list(), [-1])
test_eq(evts['source_idx'].unique().to_list(), [0])
test_eq(set(prof['key'].unique().to_list()), {'gender','occupation','age'})
test_eq(set(evts['key'].unique().to_list()), {'movie_id','rating'})
test_eq((prof['val_id'] != tok.val_vocab['[UNK]']).all(), True)
test_eq((evts['val_id'] != tok.val_vocab['[UNK]']).all(), True)

In [ ]:
u = 1
prof.filter(pl.col('user_id') == u).sort('key')

user_id,key,key_id,value,val_id,val_pos,vtype,time,logsec,source_idx,event_idx,hour,dow,dom,shard_id
i64,str,i64,str,i64,i64,str,i64,f64,i32,i64,f32,f32,f32,u64
1,"""age""",5,"""24""",1711,0,"""num""",0,0.0,-1,-1,0.0,0.0,0.0,0
1,"""gender""",6,"""M""",1686,0,"""cat""",0,0.0,-1,-1,0.0,0.0,0.0,0
1,"""occupation""",8,"""technician""",1706,0,"""cat""",0,0.0,-1,-1,0.0,0.0,0.0,0


In [ ]:
evts.filter(pl.col('user_id') == u).sort('event_idx','key','val_pos').head(12)

user_id,key,key_id,value,val_id,val_pos,vtype,time,logsec,source_idx,event_idx,hour,dow,dom,shard_id
i64,str,i64,str,i64,i64,str,i64,f64,i32,i64,f32,f32,f32,u64
1,"""movie_id""",7,"""168""",759,0,"""cat""",874965478,115.438142,0,0,21.0,1.0,22.0,0
1,"""rating""",9,"""5""",1716,0,"""num""",874965478,115.438142,0,0,21.0,1.0,22.0,0
1,"""movie_id""",7,"""172""",767,0,"""cat""",874965478,115.438142,0,1,21.0,1.0,22.0,0
1,"""rating""",9,"""5""",1716,0,"""num""",874965478,115.438142,0,1,21.0,1.0,22.0,0
1,"""movie_id""",7,"""165""",726,0,"""cat""",874965518,115.438121,0,2,21.0,1.0,22.0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1,"""rating""",9,"""4""",1713,0,"""num""",874965556,115.4381,0,3,21.0,1.0,22.0,0
1,"""movie_id""",7,"""196""",793,0,"""cat""",874965677,115.438035,0,4,22.0,1.0,22.0,0
1,"""rating""",9,"""5""",1716,0,"""num""",874965677,115.438035,0,4,22.0,1.0,22.0,0


In [ ]:
chk = evts.filter(pl.col('user_id') == 1, pl.col('key') == 'movie_id').sort('time')
chk.select('user_id','event_idx','time','logsec','hour','dow','dom').head(), chk.select('user_id','event_idx','time','logsec','hour','dow','dom').tail()

(shape: (5, 7)
 ┌─────────┬───────────┬───────────┬────────────┬──────┬─────┬──────┐
 │ user_id ┆ event_idx ┆ time      ┆ logsec     ┆ hour ┆ dow ┆ dom  │
 │ ---     ┆ ---       ┆ ---       ┆ ---        ┆ ---  ┆ --- ┆ ---  │
 │ i64     ┆ i64       ┆ i64       ┆ f64        ┆ f32  ┆ f32 ┆ f32  │
 ╞═════════╪═══════════╪═══════════╪════════════╪══════╪═════╪══════╡
 │ 1       ┆ 0         ┆ 874965478 ┆ 115.438142 ┆ 21.0 ┆ 1.0 ┆ 22.0 │
 │ 1       ┆ 1         ┆ 874965478 ┆ 115.438142 ┆ 21.0 ┆ 1.0 ┆ 22.0 │
 │ 1       ┆ 2         ┆ 874965518 ┆ 115.438121 ┆ 21.0 ┆ 1.0 ┆ 22.0 │
 │ 1       ┆ 3         ┆ 874965556 ┆ 115.4381   ┆ 21.0 ┆ 1.0 ┆ 22.0 │
 │ 1       ┆ 4         ┆ 874965677 ┆ 115.438035 ┆ 22.0 ┆ 1.0 ┆ 22.0 │
 └─────────┴───────────┴───────────┴────────────┴──────┴─────┴──────┘,
 shape: (5, 7)
 ┌─────────┬───────────┬───────────┬───────────┬──────┬─────┬──────┐
 │ user_id ┆ event_idx ┆ time      ┆ logsec    ┆ hour ┆ dow ┆ dom  │
 │ ---     ┆ ---       ┆ ---       ┆ ---       ┆ ---  ┆ --- ┆

In [ ]:
chk = evts.filter(pl.col('user_id') == 1, pl.col('key') == 'movie_id')
test_eq(chk.filter(pl.col('time') == pl.col('time').max())['logsec'].unique().item(), 0.0)
test_eq((chk['logsec'] >= 0).all(), True)
test_eq(chk.sort('time')['logsec'].to_list()[0] >= chk.sort('time')['logsec'].to_list()[-1], True)

can you check these?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

These checks look good overall. The important invariants are passing:

- Tokenized files were written correctly:
  - 4 shard parquet files
  - `tokenizer.json`

- Tokenizer looks reasonable for ML100k:
  - `10` keys
  - `1719` value tokens
  - numeric bounds for `age` and `rating`
  - no BPE needed, since there are no high-cardinality text fields

- Shape checks make sense:
  - profile: `943 users × 3 fields = 2829`
  - events: `100000 ratings × 2 fields = 200000`

- Source conventions look correct:
  - profile rows have `source_idx=-1`, `event_idx=-1`
  - event rows have `source_idx=0`
  - profile keys are `gender`, `occupation`, `age`
  - event keys are `movie_id`, `rating`

- No `[UNK]` values in this toy dataset, which is what we'd hope after fitting/tokenizing the same sources

- Time encoding looks right:
  - user's latest event has `logsec=0`
  - earlier events have positive `logsec`
  - `logsec` decreases as event time approaches the latest event
  - calendar features are populated for event rows and zero-filled for profile rows

One small thing to note: identical timestamps produce multiple `event_idx`s with the same `time` and same `logsec`, e.g. user 1 has multiple ratings at `874965478`. That's fine — `_melt_events` uses `_row_id` as a tie-breaker, so event order is deterministic even when timestamps tie.

I think this is a solid checkpoint. The next natural piece would be a small inspection method like `show_entity` or `show_batch` to make these parquet outputs easy to review without manually filtering.

so we have everything to prepare the dataset as per the paper

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — for the **data preparation / tokenization layer**, we now have the main pieces aligned with the paper:

- source schemas via `DataSource`
- profile/event/lifelong conversion to uniform KVT rows
- categorical values as single value tokens
- numerical values as percentile bucket tokens, including zero bucket
- text values as either low-cardinality categorical tokens or BPE subwords
- shared key vocab
- shared value vocab
- event time as log-seconds to latest event
- event calendar features: hour, day-of-week, day-of-month
- profile regular fields with `logsec=0`
- lifelong profile fields with log-seconds to eval point
- shared tokenizer fit across all sources
- sharded parquet output with profile and event rows

What we **do not yet have** is the later dataloader/model-prep layer:

- group tokenized rows into per-entity profile sequences
- group event rows into per-event token sequences
- insert `[USR]` and `[EVT]` tokens
- construct tensors: `key_id`, `val_id`, `val_pos`, `logsec`, calendar features
- apply max tokens per event / max profile tokens / max events per entity
- batch or pack variable-length event tokens
- optionally build MLM masks

So yes: dataset preparation is in good shape. Next stage is turning the flat tokenized parquet into model-ready samples.

summarize things in this notebook that are related todata that are not done as of now but are kept for later

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Things related to data that we've intentionally **kept for later**:

- **Truncation**
  - Not done during tokenization.
  - Kept for the dataloader so we can change experiment limits without re-tokenizing.
  - Later knobs:
    - max events per entity
    - max tokens per event
    - max profile tokens

- **Grouping flat rows into samples**
  - Current output is flat parquet rows.
  - Later we need to group by entity:
    - profile rows → one profile token sequence
    - event rows → per-event token sequences
    - event sequences → per-user history

- **Adding special tokens**
  - Tokenizer vocab already contains `[USR]` and `[EVT]`.
  - But we haven't inserted them into actual sequences yet.
  - Later:
    - prepend `[USR]` to profile sequence
    - prepend `[EVT]` to each event sequence

- **Tensor assembly**
  - We currently have parquet columns like:

```python
user_id, key_id, val_id, val_pos, logsec, hour, dow, dom, event_idx, source_idx
```

  - Later dataloader should turn these into tensors for the model.

- **Padding / packing**
  - Not implemented yet.
  - Paper uses sequence packing for variable-length event tokens.
  - Later we decide whether to:
    - pad first for simplicity
    - implement packed variable-length attention later

- **Batching**
  - Current tokenizer writes entity-sharded parquet.
  - Later dataloader should create batches, ideally with dynamic token budgets.

- **MLM masking**
  - Not implemented yet.
  - Paper uses:
    - token-level masking
    - event-level masking
    - key-level masking
    - some `[UNK]` replacement
  - This belongs after sample assembly, not in raw tokenization.

- **Train/valid/test splitting**
  - Not handled yet.
  - Tokenizer can fit on train and then encode all splits, but split logic itself still needs to be added.

- **Evaluation point handling beyond a fixed value**
  - We currently pass one `eval_time`.
  - Later real datasets may have per-entity or per-row eval points.
  - Lifelong deltas should then use the correct eval point per entity.

- **More scalable fitting**
  - Current `fit()` collects KVT into memory.
  - Fine for MovieLens and small tests.
  - Later for large data, fitting should avoid full collect where possible:
    - streaming unique cats/texts
    - approximate quantiles
    - sampled BPE training

- **Persisting tokenized dataset metadata**
  - We save the tokenizer.
  - Later we should also save dataset metadata:
    - source schema
    - shard count
    - entity column
    - feature columns
    - eval time/split info
    - version/config used for tokenization

So the notebook has completed the **flat tokenized parquet preparation**, but leaves **sample construction, truncation, batching, masking, splitting, and large-scale fitting** for the next stage.